# Chinese world — per-polity vs per-CV-token comparison (chronological pairing).

## Configuration

In [ ]:
from collections import defaultdict
import polars as pl
import duckdb

CULTURA_DB_PATH        = '../data/humans_clean.duckdb'
CROSS_VERIFIED_DB_PATH = '../data/similar_databases/cross_verified.duckdb'

## Connect

In [2]:
cultura_con        = duckdb.connect(CULTURA_DB_PATH,        read_only=True)
cross_verified_con = duckdb.connect(CROSS_VERIFIED_DB_PATH, read_only=True)

## Pairing of Cultura polities to Cross-Verified `string_citizenship_raw_d` tokens  (chronological)

In [3]:
CHINESE_POLITY_TOKEN_PAIRS = [
    (None,                              ['Xia_dynasty']),
    ('Shang Dynasty',                   ['Shang_dynasty']),
    ('Zhou Dynasty',                    ['Western_Zhou', 'Zhou_dynasty']),
    ('Qin Dynasty',                     ['Qin_Dynasty']),
    ('Han Dynasty',                     ['Han_Dynasty', 'Western_Han_Dynasty', 'Eastern_Han_Dynasty']),
    ('Xin Dynasty',                     ['Xin_dynasty']),
    ('Cao Wei',                         ['Cao_Wei']),
    ('Shu Han',                         ['Shu_Han']),
    ('Eastern Wu',                      ['Eastern_Wu']),
    ('Western Jin',                     ['Western_Jin_dynasty', 'Jin_dynasty', 'Jin_Dynasty']),
    ('Eastern Jin',                     ['Eastern_Jin_dynasty']),
    ('Northern Wei',                    ['Northern_Wei']),
    ('Sui Dynasty',                     ['Sui_Dynasty']),
    ('Tang Dynasty',                    ['Tang_Dynasty']),
    ('Five Dynasties and Ten Kingdoms', ['Five_Dynasties_and_Ten_Kingdoms']),
    ('Liao Dynasty',                    ['Liao_Dynasty']),
    ('Northern Song',                   ['Northern_Song_Dynasty', 'Song_Dynasty']),
    ('Southern Song',                   ['Southern_Song_Dynasty']),
    ('Western Xia',                     ['Western_Xia']),
    ('Yuan Dynasty',                    ['Yuan_Dynasty']),
    ('Ming Dynasty',                    ['Ming_Dynasty']),
    ('Qing Dynasty',                    ['Qing_Dynasty']),
    (None,                              ['Republic_of_China_(1912–1949)']),
    (None,                              ['China']),
    (None,                              ['Taiwan']),
]

print(f'pairs: {len(CHINESE_POLITY_TOKEN_PAIRS)}')

pairs: 25


## Cultura — qids per Chinese polity

In [4]:
cliopatria_long = cultura_con.execute("""
    SELECT  wikidata_id, polity_name
    FROM    individuals_cliopatria
    WHERE   polity_name IS NOT NULL
""").pl().with_columns(
    pl.col('polity_name').str.split(';').alias('polity')
).explode('polity').with_columns(
    pl.col('polity').str.strip_chars()
).filter(pl.col('polity') != '')

polity_to_qids = {
    polity: set(qids)
    for polity, qids in cliopatria_long.group_by('polity').agg(pl.col('wikidata_id')).iter_rows()
}

print(f'distinct polities: {len(polity_to_qids):,}')

distinct polities: 1,331


## Cross-Verified — qids per `string_citizenship_raw_d` token

In [5]:
cv_raw_citizenship = cross_verified_con.execute("""
    SELECT  wikidata_code, string_citizenship_raw_d
    FROM    individuals
    WHERE   string_citizenship_raw_d IS NOT NULL
      AND   string_citizenship_raw_d != ''
""").pl()

TOKEN_JOINER = "'_'"

def tokenize_cv_citizenship(raw):
    raw = raw.strip()
    if raw.startswith("'") and raw.endswith("'"):
        raw = raw[1:-1]
    return [token for token in raw.split(TOKEN_JOINER) if token]

token_to_qids = defaultdict(set)
for qid, raw in zip(cv_raw_citizenship['wikidata_code'].to_list(),
                    cv_raw_citizenship['string_citizenship_raw_d'].to_list()):
    if not qid:
        continue
    for token in tokenize_cv_citizenship(raw):
        token_to_qids[token].add(qid)

print(f'distinct CV tokens: {len(token_to_qids):,}')

distinct CV tokens: 1,784


## Build the polity ↔ token comparison table

In [6]:
polity_token_rows = []
all_cultura_qids  = set()
all_cv_qids       = set()

for polity, tokens in CHINESE_POLITY_TOKEN_PAIRS:
    cultura_qids_for_polity = polity_to_qids.get(polity, set()) if polity else set()
    if polity:
        all_cultura_qids |= cultura_qids_for_polity

    if not tokens:
        polity_token_rows.append({
            'cultura_polity':  polity or '',
            'cultura_n':       len(cultura_qids_for_polity) if polity else None,
            'cv_token':        '',
            'cv_n':            None,
        })
        continue

    for index, token in enumerate(tokens):
        cv_qids_for_token = token_to_qids.get(token, set())
        all_cv_qids |= cv_qids_for_token
        polity_token_rows.append({
            'cultura_polity':  (polity if polity else '') if index == 0 else '',
            'cultura_n':       (len(cultura_qids_for_polity) if polity else None) if index == 0 else None,
            'cv_token':        token,
            'cv_n':            len(cv_qids_for_token),
        })

polity_token_table = pl.DataFrame(polity_token_rows)

print('shape:', polity_token_table.shape)
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=200, fmt_str_lengths=80):
    print(polity_token_table)

shape: (31, 4)
shape: (31, 4)
┌─────────────────────────────────┬───────────┬─────────────────────────────────┬──────┐
│ cultura_polity                  ┆ cultura_n ┆ cv_token                        ┆ cv_n │
│ ---                             ┆ ---       ┆ ---                             ┆ ---  │
│ str                             ┆ i64       ┆ str                             ┆ i64  │
╞═════════════════════════════════╪═══════════╪═════════════════════════════════╪══════╡
│                                 ┆ null      ┆ Xia_dynasty                     ┆ 13   │
│ Shang Dynasty                   ┆ 7         ┆ Shang_dynasty                   ┆ 6    │
│ Zhou Dynasty                    ┆ 11        ┆ Western_Zhou                    ┆ 2    │
│                                 ┆ null      ┆ Zhou_dynasty                    ┆ 4    │
│ Qin Dynasty                     ┆ 26        ┆ Qin_Dynasty                     ┆ 9    │
│ Han Dynasty                     ┆ 1064      ┆ Han_Dynasty                     

## Polity-token totals  (union counts and overlap)

In [7]:
polity_token_totals = pl.DataFrame({
    'metric': [
        'unique Cultura individuals (union of polity rows)',
        'unique Cross-Verified individuals (union of token rows)',
        'Q-id overlap',
        'Cultura only',
        'Cross-Verified only',
    ],
    'individuals': [
        len(all_cultura_qids),
        len(all_cv_qids),
        len(all_cultura_qids & all_cv_qids),
        len(all_cultura_qids - all_cv_qids),
        len(all_cv_qids - all_cultura_qids),
    ],
})

print('shape:', polity_token_totals.shape)
polity_token_totals

shape: (5, 2)


metric,individuals
str,i64
"""unique Cultura individuals (un…",85657
"""unique Cross-Verified individu…",3618
"""Q-id overlap""",1063
"""Cultura only""",84594
"""Cross-Verified only""",2555


## Close connections

In [8]:
cultura_con.close()
cross_verified_con.close()